# ⚡ New Energy Agent · vLLM (fast ~40 tok/s)
**Colab Free T4 GPU · Runtime → Run all**

| Component | Version | Notes |
|-----------|---------|-------|
| GPU | T4 15GB | Turing SM75 (no BF16!) |
| CUDA | 12.1 | Pre-installed |
| Python | 3.12 / numpy 1.26.4 | Pre-installed |
| Inference | **vLLM 0.6.x** | float16 + XFORMERS |
| Model | Qwen2.5-3B-AWQ | ~3.5 GB VRAM |

### T4 Compatibility Fixes (auto-applied)
- `--dtype float16` — T4 has no BF16 hardware
- `VLLM_ATTENTION_BACKEND=XFORMERS` — FlashAttention not on Turing
- `--enforce-eager` — avoid CUDA Graph OOM on T4

### Usage
1. **Runtime → Run all** → Allow Drive access
2. First run ~8 min, subsequent ~2 min

💾 Model/cache on Drive, survives disconnect
🔄 Slow startup? Use `colab_notebook_transformers.ipynb` fallback


In [ ]:
# Cell 1: Read Secrets
import os
try:
    from google.colab import userdata
    for name in ['HF_TOKEN', 'NGROK_TOKEN']:
        try:
            val = userdata.get(name)
            if val:
                os.environ[name] = val
                print(f'OK {name}')
            else:
                print(f'Skip {name}')
        except: print(f'Skip {name}')
except ImportError:
    print('Not Colab')
print('Done')


In [ ]:
# Cell 2: Mount Google Drive
import os, shutil
mp = '/content/drive'
if os.path.isdir(mp) and os.listdir(mp):
    if os.path.isdir(os.path.join(mp, 'MyDrive')):
        print('Already mounted')
    else:
        for item in os.listdir(mp):
            p = os.path.join(mp, item)
            try: (shutil.rmtree if os.path.isdir(p) else os.remove)(p)
            except: pass
        from google.colab import drive; drive.mount(mp)
else:
    from google.colab import drive; drive.mount(mp)

DIRS = {
    'hf': '/content/drive/MyDrive/hf_cache',
    'vllm_kernel': '/content/drive/MyDrive/vllm_cache',
    'data': '/content/drive/MyDrive/new-energy-data',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = DIRS['hf']
os.environ['HF_HUB_CACHE'] = DIRS['hf']
os.environ['VLLM_CACHE_DIR'] = DIRS['vllm_kernel']
os.environ['NEW_ENERGY_DATA_DIR'] = DIRS['data']

import sqlite3
db = os.path.join(DIRS['data'], 'electricity_cache.db')
if os.path.exists(db):
    n = sqlite3.connect(db).execute('SELECT COUNT(*) FROM electricity_prices').fetchone()[0]
    print(f'Electricity cache: {n} records')
else:
    print('Electricity cache: empty')
print('Drive ready')


In [ ]:
# Cell 3: Install vLLM (CUDA 12 compatible) + Gradio
import subprocess, sys
print('Installing vLLM + Gradio (~3 GB, please wait)...')
# vLLM 0.6.x = CUDA 12 (Colab T4). 0.7+ = CUDA 13.
# Keep Colab default numpy 1.26.4 — DO NOT change it.
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'vllm>=0.6.3,<0.7.0',  # CUDA 12 compatible
    'gradio', 'plotly', 'pandas', 'duckduckgo_search',
    'pyngrok', 'huggingface_hub', 'openai', 'httpx', 'requests'
], check=False)
print('Done')


In [ ]:
# Cell 4: Clone repo
import os
rd = '/content/new-energy-agent'
if os.path.isdir(rd):
    %cd {rd}
    !git pull -q
else:
    !git clone -q https://github.com/pai-pixel/new-energy-agent.git {rd}
    %cd {rd}
print(f'Repo: {os.getcwd()}')


In [ ]:
# Cell 5: Download model to Drive (~2.5 GB, first time only)
import os, glob, time
from huggingface_hub import snapshot_download

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct-AWQ'
CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(CACHE, exist_ok=True)

hub = os.path.join(CACHE, 'hub')
found = None
if os.path.isdir(hub):
    dirs = glob.glob(os.path.join(hub, 'models--Qwen*', 'snapshots', '*'))
    for d in dirs:
        if os.path.isdir(d) and os.listdir(d):
            found = d; break

if found:
    gb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(found) for f in fs) / 1e9
    print(f'Model cached: {found} ({gb:.1f} GB)')
else:
    print(f'Downloading {MODEL_ID}...')
    t0 = time.time()
    found = snapshot_download(MODEL_ID, cache_dir=CACHE, resume_download=True, max_workers=4)
    print(f'Done ({time.time()-t0:.0f}s)')

os.environ['MODEL_PATH'] = found
print(f'Model: {found}')


In [ ]:
# Cell 6: Start vLLM with T4 compatibility fixes
import subprocess, os, time, sys

# T4 = Turing SM75 = NO BF16, NO FlashAttention-2
# Fix 1: --dtype float16  (T4 has no BF16 hardware)
# Fix 2: VLLM_ATTENTION_BACKEND=XFORMERS  (FlashAttention not on Turing)
# Fix 3: --enforce-eager  (avoid CUDA Graph OOM on limited VRAM)
os.environ['VLLM_ATTENTION_BACKEND'] = 'XFORMERS'

# GPU check
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
if not gpu:
    print('No GPU! Runtime -> Change runtime type -> T4 GPU')
    raise SystemExit(1)
print(f'GPU: {gpu[0]}')

model_path = os.environ.get('MODEL_PATH', '')
if not model_path:
    print('MODEL_PATH not set! Re-run Cell 5.')
    raise SystemExit(1)

print(f'Model: {model_path}')
print('Starting vLLM (first time compiles CUDA kernels ~2 min)...')

!pkill -f vllm.entrypoints 2>/dev/null || true
time.sleep(1)

logf = '/tmp/vllm.log'
with open(logf, 'w') as f:
    proc = subprocess.Popen([
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', model_path,
        '--dtype', 'float16',        # T4: no BF16
        '--enforce-eager',           # T4: avoid CUDA Graph OOM
        '--max-model-len', '4096',
        '--gpu-memory-utilization', '0.85',
        '--port', '8000',
        '--host', '0.0.0.0',
    ], stdout=f, stderr=f, env={**os.environ})

print(f'vLLM PID: {proc.pid}')
print('Waiting for vLLM to be ready...')
print(f'Monitor: !tail -f {logf}')

# Wait for ready
from openai import OpenAI
client = OpenAI(base_url='http://localhost:8000/v1', api_key='x')
ready = False
for i in range(60):
    time.sleep(5)
    try:
        client.models.list()
        ready = True
        break
    except Exception:
        if i % 6 == 0:
            print(f'  ... {i*5}s')

if not ready:
    print('vLLM failed! Last 30 lines of log:')
    !tail -30 {logf}
    raise SystemExit(1)

print('vLLM ready!')


In [ ]:
# Cell 7: Start Agent (vLLM backend)
import os, sys
%cd /content/new-energy-agent
sys.path.insert(0, '/content/new-energy-agent')

# Select vLLM backend
os.environ['INFERENCE_BACKEND'] = 'vllm'

import gradio as gr
from src.agent import NewEnergyAgent, create_ui, start_ngrok
from IPython.display import display, Javascript
import logging
logging.basicConfig(level=logging.WARNING)

ngrok_url = start_ngrok(7860)
agent = NewEnergyAgent()
demo = create_ui(agent)

print('=' * 50)
print('  New Energy Agent Ready! (vLLM ~40 tok/s)')
print('=' * 50)
if ngrok_url:
    print(f'  ngrok: {ngrok_url}')
    display(Javascript(f'window.open("{ngrok_url}", "_blank");'))
print('  Gradio URL: see cell output below for .gradio.live')
print('=' * 50)

demo.queue(max_size=32).launch(
    server_name='0.0.0.0', server_port=7860,
    share=True, show_error=True,
    css='.gradio-container{max-width:900px!important}',
    theme=gr.themes.Soft(primary_hue='green'),
)


In [ ]:
# Cell 8 (optional): Keep-alive
from IPython.display import display, Javascript
display(Javascript('setInterval(function(){document.querySelector("colab-connect-button").click()},60000)'))

import os, sqlite3
db = os.path.join(os.environ.get('NEW_ENERGY_DATA_DIR','/content/drive/MyDrive/new-energy-data'), 'electricity_cache.db')
if os.path.exists(db):
    c = sqlite3.connect(db)
    t = c.execute('SELECT COUNT(*) FROM electricity_prices').fetchone()[0]
    ps = c.execute('SELECT DISTINCT province FROM electricity_prices').fetchall()
    c.close()
    print(f'Cache: {t} records | provinces: {", ".join(p[0] for p in ps)}')
print('Keep-alive active. Model+cache on Drive.')


---
### Usage
- `Shanghai feed-in tariff` / `Jiangsu desulfurized coal price`
- `Beijing weather` / `Solar subsidy policy`
- Multi-turn: `Shanghai feed-in` -> `Jiangsu too` -> `commercial instead`

### If vLLM fails
Switch to transformers backend: open `colab_notebook_transformers.ipynb`

[GitHub](https://github.com/pai-pixel/new-energy-agent)